In [1]:
import requests
import lxml.html
from pypdf import PdfReader
import pandas as pd
from google import genai
from pathlib import Path
from google.genai import types
from ingest_sc_cases import case_df

In [ ]:
resp = requests.get("https://statecourtreport.org/state-case-database")
root = lxml.html.fromstring(resp.text)

In [ ]:
root.xpath("//div[@class = 'card card--case grid__item']")

In [3]:
case_df[case_df["state"] == "Louisiana"]

,docket_no,state,title,date,type,pending,opinion_link
417,425A21-3,Louisiana,Hoke County Board of Education v. State of Nor...,"April 2, 2026",Education,False,https://statecourtreport.org/sites/default/fil...
418,3 WAP 2024,Louisiana,Commonwealth v. Lee,"March 26, 2026",Criminal Law,False,https://statecourtreport.org/sites/default/fil...
419,SC101412,Louisiana,Luther v. Hoskins,"March 24, 2026",Voting Rights and Elections,False,https://statecourtreport.org/sites/default/fil...
420,2025-O-00879,Louisiana,In re Judge Jennifer Medley,"October 24, 2025",Judicial Selection and Administration,False,https://statecourtreport.org/sites/default/fil...
421,2024-KK-00737,Louisiana,State v. Davieontray Breax,"May 9, 2025",Criminal Law,False,https://statecourtreport.org/sites/default/fil...
422,"2024-CA-00995, 2024-CA-00996",Louisiana,"Fremin v. Boyd Racing, LLC","March 21, 2025",Civil Rights,False,https://statecourtreport.org/sites/default/fil...
423,2024-CC-00899,Louisiana,Welch v. United Medical Healthwest-New Orleans,"March 21, 2025",Voting Rights and Elections,False,https://statecourtreport.org/sites/default/fil...
424,2024-CD-00359,Louisiana,Fisher v. Harter,"October 25, 2024",Government Structure,False,https://statecourtreport.org/sites/default/fil...
425,2024-KK-00564,Louisiana,State v. Thompson,"May 2, 2023",Criminal Law,False,https://statecourtreport.org/sites/default/fil...
426,2024-C-00055,Louisiana,Watson Memorial Spiritual Temple of Christ v. ...,"June 28, 2024",Government Structure,False,https://statecourtreport.org/sites/default/fil...


In [17]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,docket_no,state,title,date,type,pending,opinion_link
5,SC-2023-0601,Alabama,Ex parte Jackson Hospital & Clinic,"October 4, 2024",Criminal Law,False,None
55,CV2024-034624,Arizona,Reuss v. Arizona,"March 5, 2025",Reproductive Rights,False,None
56,CV-24-0180-AP/EL,Arizona,Arizona Right to Life v. Fontes,"August 20, 2024",Reproductive Rights,False,None
245,1CCV-24-0000269,Hawaii,Navahine F. v. Hawaii Department of Transporta...,"June 20, 2024",Environment,False,None
265,CV01-23-14744,Idaho,Adkins v. State,"April 11, 2025",Reproductive Rights,False,None
478,SJC-13666,Massachusetts,Gotay v. Creen,"March 21, 2025",Civil Due Process,False,None
668,S-24-563,Nebraska,State ex rel. Spung v. Evnen,"October 16, 2024",Criminal Law,False,None
677,22 OC 00022 1 B,Nevada,Persaud-Zamora v. Cegavske,"April 26, 2022",Voting Rights and Elections,False,None
680,A-23-876702-W,Nevada,Silver State Hope Fund v. Nevada Department of...,"August 8, 2024",Reproductive Rights,False,None
682,88263,Nevada,Fair Maps Nevada v. Jeng,"May 10, 2024",Election 2024,False,None


In [40]:
with open("prompt.txt", "r") as prompt_file:
    prompt = prompt_file.read()

In [ ]:
gemini_key = "key"

client = genai.Client(api_key=gemini_key)
model_id = "gemini-2.5-flash-lite"

In [41]:
# Iterating through the first 5 cases for Louisiana
lou = case_df[case_df["state"] == "Louisiana"]

file_dic = {}
for i in range(5):
    print(f"Querying for case {i}")
    docket_no = lou.iloc[i]["docket_no"].replace(" ", "-")
    state = lou.iloc[i]["state"]
    date = lou.iloc[i]["date"].replace(" ", "-")

    case_ref = "_".join([docket_no, state, date])
    file_dic[case_ref] = {}

    pdf_link = lou.iloc[i]["opinion_link"]
    file_dic[case_ref]["pdf_link"] = pdf_link

    resp = requests.get(pdf_link).content

    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
    )

    file_dic[case_ref]["response"] = genai_resp.text

Querying for case 0
Querying for case 1
Querying for case 2
Querying for case 3
Querying for case 4


In [42]:
for key in file_dic.keys():
    print(f"Individual opinions for {key}")
    print(file_dic[key]["response"])

Individual opinions for 425A21-3_Louisiana_April-2,-2026
Issue being debated: Whether the trial court had subject matter jurisdiction to enter an order on April 17, 2023, when the original claims had been transformed into significantly different claims without proper amendment.

Court decision: The Supreme Court held that the trial court lacked subject matter jurisdiction to consider the transformed claims, vacated the trial court's order, and dismissed the action with prejudice.

1. Justice NEWBY: Not available
2. Justice BERGER: Concurring
3. Justice EARLS: Dissenting
4. Justice RIGGS: Dissenting
5. Justice DIETZ: Dissenting
Individual opinions for 3-WAP-2024_Louisiana_March-26,-2026
Issue being debated: Whether Pennsylvania's mandatory life sentence without parole for felony murder violates the Eighth Amendment of the U.S. Constitution (pro: 1) or Article I, Section 13 of the Pennsylvania Constitution (con: 3).

Court decision: The mandatory life sentence without parole for felony m